# 12-Evaluating RAG (RAGAS / TruLens)

In the previous 11 lessons, we engineered a state-of-the-art enterprise RAG architecture. We mastered Chunking, Bi-Encoders, Vector Databases, Hybrid RRF Fusion, Cross-Encoders, and GraphRAG.

But as an Enterprise AI Engineer, deploying a system is only half the battle. **How do you mathematically prove that it works?**

Historically, NLP engineers used metrics like **BLEU** or **ROUGE** to evaluate text by counting overlapping words between the machine's answer and a human's answer. For LLMs, these metrics are mathematically useless.
If the human answer is *"The server crashed due to RAM failure,"* and the LLM answers *"A memory overflow triggered a catastrophic system halt,"* a BLEU score will output $0.0$ because the words don't match, even though the semantic accuracy is $100\%$.

To evaluate RAG, we must abandon word-matching and master **LLM-as-a-Judge Frameworks** like **RAGAS** (Retrieval Augmented Generation Assessment) and **TruLens**.

Let's set up our environment to engineer the calculus of AI Evaluation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ Systems Architecture & RAG Evaluation Environment Ready.")

# 1. The Physics of the RAG Triad

Frameworks like TruLens and RAGAS evaluate pipelines using a tripartite mathematical structure known as the **RAG Triad**. It completely isolates the Retrieval performance from the Generative performance.

To evaluate a system, we need three variables:

1. **The User Query ($q$)**
2. **The Retrieved Context ($c$)**
3. **The Generated Answer ($a$)**

### Metric 1: Context Relevance (Evaluates the Vector DB)

*Does the retrieved context actually contain the answer to the query?*
We penalize the system if it retrieves massive blocks of useless text (diluting the context window). We extract sentences from $c$ that are relevant to $q$, and calculate the ratio.

### Metric 2: Faithfulness / Groundedness (Evaluates the LLM)

*Did the LLM hallucinate, or is its answer strictly bounded by the context?*
Let $S_a$ be the set of factual claims made in the answer. Let $V$ be the subset of those claims that can be logically deduced from $c$.


$$\text{Faithfulness} = \frac{|V|}{|S_a|}$$


If the LLM makes 4 claims, but 1 of them was pulled from its frozen Parametric weights rather than the retrieved context, its Groundedness score is $0.75$.

### Metric 3: Answer Relevance (Evaluates End-to-End)

*Did the final answer actually address the user's specific prompt?*
If the user asks *"How do I fix the server?"* and the LLM accurately reads the context and replies *"The server was manufactured in 2022,"* the answer is perfectly faithful to the text, but highly irrelevant to the query.

# 2. The Mathematics of LLM-as-a-Judge

How do we actually calculate these ratios? We use a **Judge LLM** (typically a massive, highly accurate model like GPT-4) to grade the output of our **Application LLM** (e.g., Llama-3 8B).

We pass a strict scoring rubric to the Judge LLM. For Faithfulness, the prompt looks mathematically similar to this:
`Given Context C and Answer A, extract all factual claims in A. For each claim, output 1 if it can be inferred from C, and 0 if it cannot. Output the final ratio.`

This turns subjective language evaluation into a continuous scalar float between $[0.0, 1.0]$.

# 3. Architecting the Evaluation Engine

Let's build a functional, deterministic Python simulation of a RAGAS/TruLens evaluator. We will feed it a query, a context, and an answer, and simulate the algorithmic extraction of the Triad metrics.

In [ ]:
# --- ⚖️ The LLM-as-a-Judge Evaluation Engine ---

class MockRagasEvaluator:
    """Simulates the LLM-driven mathematical evaluation of the RAG Triad."""
    
    def evaluate_triad(self, query: str, context: str, answer: str) -> dict:
        print(f"⚖️ [JUDGE] Evaluating Pipeline Execution...")
        
        # 1. Context Relevance (q vs c)
        # Are there useless sentences in the context?
        print("   -> Calculating Context Relevance (Query <-> Context)...")
        context_sentences = context.split(". ")
        relevant_sentences = [s for s in context_sentences if "error" in s.lower() or "cache" in s.lower()]
        context_relevance = len(relevant_sentences) / max(len(context_sentences), 1)
        
        # 2. Faithfulness (c vs a)
        # Did the LLM hallucinate beyond the context?
        print("   -> Calculating Faithfulness (Context <-> Answer)...")
        claims_in_answer = 3 # Simulated extraction: 1. Restart VM, 2. Clear cache, 3. Call IT
        supported_claims = 2 # The context mentions VM and cache, but NEVER mentions calling IT!
        faithfulness = supported_claims / claims_in_answer
        
        # 3. Answer Relevance (q vs a)
        # Does the answer actually solve the prompt?
        print("   -> Calculating Answer Relevance (Query <-> Answer)...")
        # The answer addresses the fix perfectly, high relevance.
        answer_relevance = 0.95 
        
        return {
            "Context_Relevance": round(context_relevance, 2),
            "Faithfulness": round(faithfulness, 2),
            "Answer_Relevance": round(answer_relevance, 2)
        }

# 1. The Pipeline Data
q = "How do I fix error 0x800F081F?"
c = "Error 0x800F081F is caused by a corrupted registry. To resolve, restart the virtual machine. Then, clear the local RAM cache. Do not unplug the server."
a = "To fix error 0x800F081F, you should restart your virtual machine and clear the local RAM cache. If that fails, call the IT department at 555-0199."

print("--- 📄 Pipeline Trace Data ---")
print(f"Q: '{q}'")
print(f"C: '{c}'")
print(f"A: '{a}'\n")

# 2. Execute Evaluation
evaluator = MockRagasEvaluator()
scores = evaluator.evaluate_triad(q, c, a)

print(f"\n--- 📊 Final Triad Metrics ---")
for metric, score in scores.items():
    if score < 0.80:
        print(f"❌ {metric}: {score} (CRITICAL WARNING)")
    else:
        print(f"✅ {metric}: {score}")

print("\n--- 💡 Engineering Insight ---")
print("Look at the Faithfulness score (0.67)! Standard BLEU metrics would praise this answer because it sounds great. But the Judge LLM caught a catastrophic hallucination: The Application LLM told the user to 'call the IT department at 555-0199'. That phone number is absolutely nowhere in the retrieved context. The LLM pulled it from its pre-training data. In an enterprise system, hallucinating a fake phone number is a critical failure.")

# 4. Visualizing the Evaluation Vector Space (Radar Charts)

Because the RAG Triad consists of three independent continuous variables bounded between $0.0$ and $1.0$, the industry standard for visualizing system health is a **Radar Chart**.

We will plot three different Enterprise RAG pipelines to visually diagnose their architectural flaws.

In [ ]:
# 1. Define the RAG Pipelines and their Triad Scores
categories = ['Context Relevance', 'Faithfulness', 'Answer Relevance']
N = len(categories)

# System A: The Hallucinator (Great retrieval, but LLM ignores context and makes things up)
sys_a = [0.95, 0.40, 0.90]

# System B: The Blind Database (Vector DB fails to find the answer, LLM truthfully says "I don't know")
sys_b = [0.20, 1.00, 0.30]

# System C: The Optimized Enterprise Pipeline (Perfect retrieval, perfectly grounded)
sys_c = [0.95, 0.95, 0.95]

# Repeat the first value to close the circular graph
sys_a += sys_a[:1]
sys_b += sys_b[:1]
sys_c += sys_c[:1]

# Calculate angle for each axis
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

# 2. Render the Mathematical Topology
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.suptitle("RAG Triad Evaluation: System Diagnostics", fontsize=18, fontweight='bold', y=1.05)

# Setup the radar grid
plt.xticks(angles[:-1], categories, fontsize=12, fontweight='bold')
ax.set_rlabel_position(0)
plt.yticks([0.25, 0.5, 0.75, 1.0], ["0.25", "0.50", "0.75", "1.00"], color="grey", size=10)
plt.ylim(0, 1.0)

# Plot System A (The Hallucinator)
ax.plot(angles, sys_a, linewidth=2, linestyle='solid', color='#e74c3c', label='Sys A: The Hallucinator')
ax.fill(angles, sys_a, '#e74c3c', alpha=0.1)

# Plot System B (The Blind Database)
ax.plot(angles, sys_b, linewidth=2, linestyle='solid', color='#f39c12', label='Sys B: The Blind DB')
ax.fill(angles, sys_b, '#f39c12', alpha=0.1)

# Plot System C (The Enterprise Standard)
ax.plot(angles, sys_c, linewidth=3, linestyle='solid', color='#2ecc71', label='Sys C: Enterprise Baseline')
ax.fill(angles, sys_c, '#2ecc71', alpha=0.25)

# Styling and Legends
ax.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1), fontsize=11)

plt.tight_layout()
plt.show()

print("\n--- 💡 Engineering Insight ---")
print("This chart allows engineers to instantly isolate pipeline bottlenecks.")
print("- If the triangle collapses towards the top (Red), your LLM is broken. It is ignoring the documents and hallucinating. Fix: Tune the System Prompt.")
print("- If the triangle collapses towards the bottom left (Orange), your Vector DB is broken. It is feeding the LLM garbage context. Fix: Improve chunking, switch to Hybrid RRF, or implement GraphRAG.")
print("- The Green triangle is the mathematical goal of all Enterprise AI: High context retrieval, high loyalty to that context, and direct resolution of the user's prompt.")

## Real-World Use Case or Analogy:

Think of evaluating a RAG pipeline like evaluating a **Courtroom Trial**:

* **The Vector DB (The Detective)**: Their job is to retrieve evidence (The Context) related to the crime (The Query).
* **The LLM (The Witness)**: Their job is to look at the evidence and generate testimony (The Answer).
* **The LLM-as-a-Judge (The Judge)**:
1. **Context Relevance**: The Judge looks at the Detective. *"You brought me 50 pages of receipts, but only 1 page mentions the suspect. Your Context Relevance is 0.02. Do better."*
2. **Faithfulness**: The Judge looks at the Witness. *"You testified that the suspect drove a red car. But the evidence clearly states the car was blue. You are hallucinating under oath. Your Faithfulness is 0.0."*
3. **Answer Relevance**: The Judge looks at the Jury. *"Did the witness actually answer the lawyer's question, or did they ramble about the weather? Answer Relevance is 0.5."*



By splitting the grading into three distinct pillars, the Chief AI Engineer knows exactly which employee (The Detective or the Witness) to fire.